# RAG Basics

## Setup

In [1]:
# from IPython.display import Markdown, display
from pprint import pprint
import openai
import httpx
import os
import numpy as np

API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError(
        "Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation."
    )

BASE_URL_CLEAN = os.getenv("MODEL_BASE_URL_CLEAN")
if not BASE_URL_CLEAN:
    raise RuntimeError(
        "Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates."
    )

# local CA from your compose cert setup
CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
VERIFY_CONFIG = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True

GENERATION_MODELS = ["ollama_chat/llama3.2:3b", "ollama_chat/llama3.2:1b"]
EMBEDDING_MODELS = [
    "ollama/nomic-embed-text:latest",
    "ollama/qllama/bge-small-en-v1.5:q4_k_m",
]

## Generate RAG content and embed

In [2]:
def create_rag_content(model, question):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
        max_retries=0,
    )
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": question}],
            temperature=0.0,
            stream=True,
        )
        print(f"Asking {model}: ", end="")

        parts = []
        for chunk in stream:
            if not getattr(chunk, "choices", None):
                continue
            delta = chunk.choices[0].delta.content
            if isinstance(delta, str):
                print(".", flush=True, end="")
                parts.append(delta)

        print(" Done.")
        return "".join(parts)
    except openai.OpenAIError as e:
        print(f"OpenAI API error creating RAG content: {e}")
        return None
    except Exception as e:
        print(f"Error creating RAG content: {e}")
        return None


def collect_rag_content(questions, generation_models):
    rag_content = []
    for question in questions:
        for model in generation_models:
            content = create_rag_content(model, question)
            if content:
                rag_content.append(content)
    return rag_content


questions = [
    "Gib genau 3 Vortragstitel zu KI aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.",
    "Gib genau 3 Vortragstitel zu Rasenpflege aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.",
    "Gib genau 3 Vortragstitel zu Rasenpflege mit KI aus. Ausgabeformat: Titel1$Titel2$Titel3. Keine weiteren Zeichen.",
]

rag_raw_content = collect_rag_content(questions, GENERATION_MODELS)
rag_content = []
for content in rag_raw_content:
    lines = content.split("$")
    for line in lines:
        line = line.strip().replace("\n", "")
        if line:
            rag_content.append(line)

pprint(rag_content)
# display(Markdown(rag_content))

Asking ollama_chat/llama3.2:3b: ....................................................... Done.
Asking ollama_chat/llama3.2:1b: ............................................ Done.
Asking ollama_chat/llama3.2:3b: ............................................................. Done.
Asking ollama_chat/llama3.2:1b: .................................................... Done.
Asking ollama_chat/llama3.2:3b: ........................................................ Done.
Asking ollama_chat/llama3.2:1b: ................................................................ Done.
['"Die Zukunft der Arbeit: Wie KI unsere Welt verändert""KI und Ethik: Die '
 'Herausforderungen einer moralischen Intelligenz""Machine Learning: Von Daten '
 'zu Entscheidungen - Die Kunst der KI-Modellierung"',
 '1. Die Zukunft der Maschinenlehrer2. Künstliche Intelligenz und soziale '
 'Verantwortung3. Der Einsatz von KI in der Gesundheitsversorgung',
 '* "Optimale Pflegemethoden für ein gesundes Gras"* "Die Kunst der '
 'Rasen

Embed RAG context

In [3]:
def embed_texts(texts, model):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
    )

    try:
        response = client.embeddings.create(
            model=model,
            input=texts,
        )
        return [item.embedding for item in response.data]

    except openai.OpenAIError as e:
        print(f"OpenAI API error embedding texts: {e}")
        return None
    except Exception as e:
        print(f"Error embedding texts: {e}")
        return None
        print(f"Error embedding texts: {e}")
        return None


embeddings = embed_texts(rag_content, EMBEDDING_MODELS[0])
if embeddings:
    print(f"Embedding: {embeddings[0]}")

Embedding: [0.029083284, 0.0230517, -0.13098134, -0.00038902016, 0.044704255, 0.0098157255, -0.041648638, -0.041365873, -0.05799925, 0.03248369, -0.03320763, 0.052039515, 0.03928382, 0.056480803, 0.013107013, -0.04735045, 0.005807499, -0.08561691, -0.03955633, 0.067576066, 0.0061014723, 0.011458503, 0.0014542557, -0.049989205, 0.11739692, 0.028425144, 0.020589618, 0.03947498, -0.0387477, -0.014452478, -0.008749317, -0.038927425, -0.01664842, 0.0051086983, -0.037721697, -0.062108047, 0.023171784, 0.11109703, -0.00017315775, -0.01679043, 0.042878088, -0.008337985, -0.029025927, -0.030688329, 0.07223849, -0.063276105, -0.0023233667, 0.01949591, 0.093058385, -0.052908782, -0.038982194, -0.030400477, 0.031179568, -0.04070722, 0.0758132, -0.00865269, 0.016365265, -0.019002395, 0.017144468, 0.02276109, 0.015088654, 0.013108155, -0.035104174, -0.0028140165, 0.017170887, -0.026651973, -0.03842798, 0.0673587, 0.03071111, 0.0110664675, 0.018578859, -0.02182578, -0.0028386507, 0.026473647, 0.01539

## Numpy distance calculation from example notebook

In [4]:
vectors = np.array(embeddings)  # (n, d)
query = np.array(
    embed_texts(["I am interested in AI talks"], EMBEDDING_MODELS[0])[0]
)  # (d,)
print("Vector shape:", vectors.shape)
print("Query shape:", query.shape)

# Compute Euclidean distances
differences = vectors - query  # shape: (n_vectors, vector_dim)
distances = np.linalg.norm(differences, axis=1)

# Index of closest vector
closest_index = np.argmin(distances)

# Sorted indices by distance
sorted_indices = np.argsort(distances)


print("Closest vector index:", closest_index)
print("Distance:", distances[closest_index])
print("Sorted indices by distance:", sorted_indices)

Vector shape: (6, 768)
Query shape: (768,)
Closest vector index: 1
Distance: 0.9578552632257035
Sorted indices by distance: [1 2 4 3 5 0]


### Closest

In [5]:
print(rag_content[closest_index])

1. Die Zukunft der Maschinenlehrer2. Künstliche Intelligenz und soziale Verantwortung3. Der Einsatz von KI in der Gesundheitsversorgung


In [6]:
# Sort festival_talks according to the sorted_indices
sorted_festival_talks = [rag_content[i] for i in sorted_indices]

# Optionally print it
for num, talk in enumerate(sorted_festival_talks):
    print(num + 1, talk)
    print()

1 1. Die Zukunft der Maschinenlehrer2. Künstliche Intelligenz und soziale Verantwortung3. Der Einsatz von KI in der Gesundheitsversorgung

2 * "Optimale Pflegemethoden für ein gesundes Gras"* "Die Kunst der Rasenpflege: von Grundlagen bis hin zu Fortgeschrittenen"* "Rasenpflege im Winter: Tipps und Tricks für eine erfolgreiche Wintersaison"

3 Optimierung der Rasenpflege mithilfe von Künstlicher IntelligenzAutomatisierte Pflegedienste für perfekte GrasflächenIntelligente Rasenpflegesysteme: Die Zukunft der Graspflege

4 Rasenpflege - Die Bedeutung der Pflege Ihres RasensRasenpflege und Gartenarbeit - Ein wichtiger Teil unseres LebensGartenreinigung - Die Notwendigkeit einer regelmäßigen Pflege

5 1. "Rasen mit KI: Eine umweltfreundliche Alternative zur traditionellen Pflege"2. "KI-gestützte Rasenpflege für effizientes und gesundes Grün"3. "Rasen mit KI: Die Zukunft der Landwirtschaft"

6 "Die Zukunft der Arbeit: Wie KI unsere Welt verändert""KI und Ethik: Die Herausforderungen einer mo

## RAG request

In [7]:
retrieved_festival_talks = sorted_festival_talks[:3]


def chat_w_rag(model, question, rag_content):
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    client = openai.OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL_CLEAN,
        http_client=http_client,
        max_retries=0,
    )
    content = (
        f"Nutzerfrage: {question}\n\nHier sind die am besten passenden Vorträge:\n\n "
        + "\n".join(rag_content)
        + "\n\nAntworte auf die Nutzerfrage"
    )
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": content}],
            temperature=0.0,
            stream=True,
        )
        print(f"\n\nAsking {model}:")

        for chunk in stream:
            if not getattr(chunk, "choices", None):
                continue
            delta = chunk.choices[0].delta.content
            if isinstance(delta, str):
                print(delta, flush=True, end="")

        print(" Done.")

    except openai.OpenAIError as e:
        print(f"OpenAI API error creating RAG content: {e}")

    except Exception as e:
        print(f"Error creating RAG content: {e}")


for model in GENERATION_MODELS:
    chat_w_rag(
        model, "Ich interessiere mich für Vorträge zu KI", retrieved_festival_talks
    )



Asking ollama_chat/llama3.2:3b:
Hallo!

Es scheint, als ob du dich für Vorträge zu Künstlicher Intelligenz (KI) interessierst und möglicherweise auch für Themen wie Rasenpflege oder Maschinenlehrer. Ich werde dir einige Vorschläge machen, um deine Interessen zu befriedigen.

**Vorträge zu KI:**

1. **"Die Zukunft der Maschinenlehrer"**: Ein interessanter Titel, der wahrscheinlich über die Entwicklung von künstlichen Intelligenz in der Bildung und im Lernen sprechen wird.
2. **"Künstliche Intelligenz und soziale Verantwortung"**: Ein Vortrag, der sich mit den ethischen Aspekten von KI auseinandersetzen wird, wie z.B. die Verantwortung von KI-Systemen gegenüber Menschen und Umwelt.
3. **"Der Einsatz von KI in der Gesundheitsversorgung"**: Ein Vortrag, der sich mit den Möglichkeiten und Herausforderungen von KI in der Medizin auseinandersetzen wird, wie z.B. die Entwicklung von künstlichen Intelligenz-Systemen für Diagnose und Behandlung.

**Rasenpflege und KI:**

Wenn du jedoch auch an